In [1]:
import pickle

In [2]:
def load_obj( name ):
    """
    Load dataset from pickle file.
    :param name: Full pathname of the pickle file
    :return: Dataset type of dictionary
    """
    with open( name , 'rb') as f:
        return pickle.load(f)

In [ ]:
dataset = load_obj('raw_pc50_STRING') 
    
#nlabels = 2
    #networks_name = ['string','CPDB','pcnet','pcnet','pcnet','pcnet','string']
        

#data = dataset[0]
#data.adj_t = data.adj_t.to_symmetric()
        

#x = data.x


In [4]:
mask = dataset['split_set']

In [5]:
mask

{0: [(array([ True, False, False, ..., False,  True, False]),
   array([False, False, False, ..., False, False, False])),
  (array([False, False, False, ...,  True,  True, False]),
   array([False, False, False, ..., False,  True, False])),
  (array([ True, False, False, ...,  True, False, False]),
   array([False,  True, False, ..., False, False, False])),
  (array([False,  True, False, ...,  True, False, False]),
   array([False,  True, False, ..., False, False, False])),
  (array([False, False, False, ..., False,  True, False]),
   array([False, False, False, ...,  True, False, False]))],
 1: [(array([False, False, False, ..., False,  True, False]),
   array([False, False, False, ...,  True, False, False])),
  (array([False, False, False, ..., False,  True, False]),
   array([False, False, False, ...,  True, False, False])),
  (array([ True, False, False, ...,  True,  True, False]),
   array([False, False, False, ..., False, False, False])),
  (array([False, False, False, ...,  True

In [6]:

from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

data = Data(x=dataset['feature'], y=dataset['label'], edge_index=dataset['edge_index'], mask=mask, node_names=dataset['node_name'])

In [7]:
dataset['node_name'][12]

'RAB11FIP3'

In [8]:
data

Data(
  edge_index=[2, 489956],
  mask={
    0=[5],
    1=[5],
    2=[5],
    3=[5],
    4=[5],
    5=[5],
    6=[5],
    7=[5],
    8=[5],
    9=[5]
  },
  node_names=[13137],
  x=[13137, 50],
  y=[13137]
)

In [9]:
data.edge_index

tensor([[    0,     1,     2,  ..., 12608,  3009,  3009],
        [   12,    12,    12,  ..., 12892, 11299, 11300]])

In [10]:
import pandas as pd
import numpy as np
import networkx as nx
import h5py, os, sys
import random
import matplotlib.pyplot as plt
import seaborn as sns

swapping_percentages = [0, 0.25, 0.5, 0.75, 1]

In [11]:
import pickle
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
import pickle
import torch
import sys

from torch_geometric.data import Data
import torch_geometric.transforms as T


In [12]:
import torch
from torch_geometric.data import Data
from torch_geometric.utils import to_networkx, from_networkx
G = to_networkx(data, to_undirected=False)

In [13]:
import random
import networkx as nx

# 获取所有边
edges = list(G.edges())

# 设置扰动比例
disturb_ratios = [0.25, 0.5, 0.75, 1]

# 存储扰动后的网络
rewired_networks = [G]

for ratio in disturb_ratios:
    # 计算需要扰动的边数
    swap_count = int(len(edges) * ratio)
    
    # 创建一个副本，以避免直接修改原始网络
    G_copy = G.copy()
    
    # 随机选择要扰动的边
    edges_to_swap = random.sample(edges, swap_count)
    
    # 删除这些边
    G_copy.remove_edges_from(edges_to_swap)
    
    # 随机添加新的边
    while len(G_copy.edges()) < len(edges) - len(edges_to_swap) + swap_count:
        u, v = random.sample(G_copy.nodes(), 2)
        if not G_copy.has_edge(u, v):  # 避免重复边
            G_copy.add_edge(u, v)
    
    # 将扰动后的网络添加到列表中
    rewired_networks.append(G_copy)



In [14]:
perturbed_networks = rewired_networks

In [16]:
fig = plt.figure(figsize=(10, 20))
bins = np.linspace(0, 200, 100)

for i in range(len(swapping_percentages)):
    plt.subplot(len(swapping_percentages), 1, i+1)
    _ = plt.hist([v for k, v in perturbed_networks[i].degree()], bins)
    plt.title("Perturbation: {}".format(swapping_percentages[i]), size=25)

plt.tight_layout()

# Save the plot to both PDF and PNG
plt.savefig("Supplementary_Figure1.pdf", format="pdf", bbox_inches='tight')
plt.savefig("Supplementary_Figure1.png", format="png", dpi=600, bbox_inches='tight')

plt.close()

In [49]:
rewired_networks

In [50]:
import networkx as nx

def get_highest_degree_node(G):
    # 获取度数最高的节点
    highest_degree_node = max(G.degree, key=lambda x: x[1])
    return highest_degree_node  # 返回 (节点, 度数)

# 示例：
# G = nx.erdos_renyi_graph(100, 0.05)  # 生成随机图
print(get_highest_degree_node(perturbed_networks[0]))  # 输出度数最高的节点


(28, 828)


In [51]:
print(get_highest_degree_node(perturbed_networks[0])) 
print(get_highest_degree_node(perturbed_networks[1])) 
print(get_highest_degree_node(perturbed_networks[2])) 
print(get_highest_degree_node(perturbed_networks[3])) 
print(get_highest_degree_node(perturbed_networks[4])) 

(28, 828)
(28, 650)
(28, 422)
(28, 229)
(2761, 64)


In [52]:
print(perturbed_networks[0].number_of_edges())
print(perturbed_networks[1].number_of_edges())
print(perturbed_networks[2].number_of_edges())
print(perturbed_networks[3].number_of_edges())
print(perturbed_networks[4].number_of_edges())

244860
244860
244860
244860
244860


In [15]:
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

def plot_degree_distribution(G, filename, ratio):
    degrees = [d for n, d in G.degree()]
    
    plt.figure(figsize=(4, 4))
    plt.hist(degrees, bins=range(1, max(degrees) + 2), color="#69b3a2", edgecolor="black", alpha=0.7)
    plt.xlabel("Degree", fontsize=14)
    plt.ylabel("Count", fontsize=14)
    plt.title(f"Perturbed Fraction: {ratio}", fontsize=16)
    plt.xticks(
        ticks=range(0, max(degrees) + 1, max(1, max(degrees) // 10)),
        fontsize=12,
        rotation=45
    )
    plt.yticks(fontsize=12)
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(f"{filename}.pdf")
    plt.close()

# 示例调用：
plot_degree_distribution(perturbed_networks[0], "degree_dist_0",'0')
plot_degree_distribution(perturbed_networks[1], "degree_dist_0.25",'0.25')
plot_degree_distribution(perturbed_networks[2], "degree_dist_0.5",'0.5')
plot_degree_distribution(perturbed_networks[3], "degree_dist_0.75",'0.75')
plot_degree_distribution(perturbed_networks[4], "degree_dist_1",'1')
